In [ ]:
import cv2
import os
from ultralytics import YOLO

# 1. Configuración del modelo y clases
ruta_pesos = '../models/entrenamiento_script_estable/weights/best.pt'
model = YOLO(ruta_pesos)
nombres_clases = {0: 'Vehiculo', 1: 'Peaton', 2: 'Dos_Ruedas', 3: 'Senales'}

# Crear carpeta para los resultados si no existe
os.makedirs('../reports/alertas_visuales', exist_ok=True)

# 2. Rutas de los videos (ACTUALIZA LA RUTA DE ENTRADA CON TU VIDEO REAL)
ruta_video_entrada = '../test/dia.mov' # Cambia esto por la ruta de tu video de prueba
ruta_video_salida = '../reports/alertas_visuales/test.mp4'

# 3. Inicializar la lectura del video con OpenCV
cap = cv2.VideoCapture(ruta_video_entrada)

if not cap.isOpened():
    print(f"Error: No se pudo abrir el video en {ruta_video_entrada}")
else:
    # Obtener propiedades del video original para replicarlas
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    area_total = ancho * alto

    # Configurar el "escritor" del nuevo video (.mp4)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(ruta_video_salida, fourcc, fps, (ancho, alto))

    print(f"Iniciando procesamiento de video: {total_frames} frames a {fps} FPS...")

    frame_count = 0

    # 4. Bucle principal: Leer y procesar cuadro por cuadro
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break # Se acabó el video

        # Hacer predicción sobre el frame actual (verbose=False para no saturar la consola)
        resultados = model(frame, verbose=False)

        # 5. Aplicar la Lógica de Riesgo Híbrida
        for resultado in resultados:
            cajas = resultado.boxes
            for caja in cajas:
                confianza = float(caja.conf[0])
                if confianza < 0.4:
                    continue
                    
                clase_id = int(caja.cls[0])
                nombre_objeto = nombres_clases[clase_id]
                x1, y1, x2, y2 = map(int, caja.xyxy[0].tolist())
                
                area_obj = (x2 - x1) * (y2 - y1)
                porcentaje = (area_obj / area_total) * 100
                
                # Reglas de semáforo de riesgo
                color = (0, 255, 0) # Verde (Bajo)
                texto_alerta = f"{nombre_objeto} {porcentaje:.1f}%"
                
                if clase_id == 0 and porcentaje > 15.0:
                    color = (0, 0, 255) # Rojo (ALTO)
                    texto_alerta = "¡COLISION INMINENTE!"
                elif clase_id in [1, 2] and porcentaje > 5.0:
                    color = (0, 0, 255) # Rojo (CRÍTICO)
                    texto_alerta = "¡PELIGRO PEATON/MOTO!"
                elif porcentaje > 8.0:
                    color = (0, 255, 255) # Amarillo (MEDIO)
                    texto_alerta = "Precaucion"

                # Dibujar las alertas sobre el frame
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                (w, h), _ = cv2.getTextSize(texto_alerta, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
                cv2.rectangle(frame, (x1, y1 - 20), (x1 + w, y1), color, -1)
                cv2.putText(frame, texto_alerta, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)

        # Escribir el frame ya procesado (con las cajas pintadas) en el nuevo video
        out.write(frame)
        
        frame_count += 1
        # Imprimir progreso cada 50 frames para que sepas que no se ha trabado
        if frame_count % 50 == 0:
            print(f"Procesando frame {frame_count}/{total_frames}...")

    # 6. Liberar recursos y cerrar
    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print(f"¡Procesamiento completado! Video guardado en: {ruta_video_salida}")

Iniciando procesamiento de video: 1215 frames a 30 FPS...
Procesando frame 50/1215...
Procesando frame 100/1215...
Procesando frame 150/1215...
Procesando frame 200/1215...
Procesando frame 250/1215...
Procesando frame 300/1215...
Procesando frame 350/1215...
Procesando frame 400/1215...
Procesando frame 450/1215...
Procesando frame 500/1215...
Procesando frame 550/1215...
Procesando frame 600/1215...
Procesando frame 650/1215...
Procesando frame 700/1215...
Procesando frame 750/1215...
Procesando frame 800/1215...
Procesando frame 850/1215...
Procesando frame 900/1215...
Procesando frame 950/1215...
Procesando frame 1000/1215...
Procesando frame 1050/1215...
Procesando frame 1100/1215...
Procesando frame 1150/1215...
Procesando frame 1200/1215...
¡Procesamiento completado! Video guardado en: ../reports/alertas_visuales/test.mp4
